# 03 — Universo de acciones candidatas (IEC15)

**Objetivo:** obtener la lista de todas las acciones que hoy cotizan en la Bolsa de Santiago y marcar las exclusiones de la sección 2 de la metodología:
- AFP
- Fondos de inversión, fondos mutuos y ETF
- Series duplicadas de una misma empresa (solo entra la más líquida; se revisa a mano)

**Limitación conocida:** el buscador de Yahoo solo entrega las acciones que cotizan **hoy**. Las que se deslistaron o fusionaron desde 2019 se agregan en un paso aparte, para controlar el sesgo de supervivencia.

La lista se guarda en `data/raw/` (excluida del repositorio). Antes de hacer commit: **Edit → Clear Outputs of All Cells**.

In [ ]:
from pathlib import Path
import pandas as pd
import yfinance as yf
from yfinance import EquityQuery

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 60)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CARPETA_RAW = RAIZ / "data" / "raw"
CARPETA_RAW.mkdir(parents=True, exist_ok=True)
print("yfinance", yf.__version__)

## 1. Buscar todas las acciones de la Bolsa de Santiago

Se usa el buscador (screener) de Yahoo Finance filtrando por la bolsa de Santiago (código `SGO`). Si ese filtro falla, se intenta por región Chile (`cl`). El buscador entrega hasta 250 resultados por consulta, así que se pide por páginas.

In [ ]:
def buscar_todo(consulta, tam=250):
    filas, offset = [], 0
    while True:
        res = yf.screen(consulta, offset=offset, size=tam)
        quotes = res.get("quotes", [])
        filas += quotes
        if len(quotes) < tam:
            return filas
        offset += tam

try:
    filas = buscar_todo(EquityQuery("eq", ["exchange", "SGO"]))
    print("Filtro usado: bolsa SGO")
except Exception as e:
    print("Filtro por bolsa falló:", e)
    filas = buscar_todo(EquityQuery("eq", ["region", "cl"]))
    print("Filtro usado: región cl")

cand = pd.DataFrame(filas)
print(len(cand), "valores encontrados")
print("Columnas disponibles:", cand.columns.tolist())

## 2. Ordenar y marcar exclusiones sugeridas

Las marcas son **sugerencias automáticas** por nombre y tipo. La decisión final se revisa a mano, porque un nombre puede engañar.

In [ ]:
columnas = ["symbol", "longName", "shortName", "quoteType", "marketCap",
            "averageDailyVolume3Month", "regularMarketPrice", "exchange"]
cand = cand[[c for c in columnas if c in cand.columns]].copy()

nombre = (cand.get("longName", pd.Series("", index=cand.index)).fillna("") + " " +
          cand.get("shortName", pd.Series("", index=cand.index)).fillna("")).str.upper()
simbolo = cand["symbol"].str.upper()

def motivo(i):
    if "quoteType" in cand.columns and cand.at[i, "quoteType"] != "EQUITY":
        return "No es acción (" + str(cand.at[i, "quoteType"]) + ")"
    if simbolo[i].startswith(("CFI", "CFM")):
        return "Fondo de inversión / mutuo / ETF"
    if "AFP" in nombre[i] or "FONDOS DE PENSIONES" in nombre[i]:
        return "AFP"
    if "FONDO" in nombre[i] or " ETF" in nombre[i]:
        return "Fondo / ETF (revisar)"
    return ""

cand["exclusion_sugerida"] = [motivo(i) for i in cand.index]

# Valor transado diario aproximado (precio x volumen promedio de 3 meses), solo para ordenar
if {"averageDailyVolume3Month", "regularMarketPrice"} <= set(cand.columns):
    cand["valor_diario_aprox_mm_clp"] = (cand["averageDailyVolume3Month"] *
                                         cand["regularMarketPrice"] / 1e6).round(1)
    cand = cand.sort_values("valor_diario_aprox_mm_clp", ascending=False)

print("Sugeridas para excluir:", (cand["exclusion_sugerida"] != "").sum())
print("Candidatas:", (cand["exclusion_sugerida"] == "").sum())
cand.reset_index(drop=True)

## 3. Guardar la lista

Se guarda completa (con las exclusiones sugeridas) para revisarla antes del paso 4b.

In [ ]:
ruta = CARPETA_RAW / "candidatos_yahoo.csv"
cand.to_csv(ruta, index=False, encoding="utf-8-sig")
print("Guardado en:", ruta)